# Week 3 — Classical Baseline FIXED

**What changed:**  
- `get_loaders_from_features` previously loaded **333 files** (221 labeled + 112 `test_*` with no ground-truth → silently mislabeled as 0).  
- Fixed: filter to `normal_*` / `tumor_*` only → **221 labeled slides** (110 normal + 111 tumor).  
- Stratified 70/15/15 split: **train=154 / val=33 / test=34**

**Same anti-overfit settings as E2:**
- `dropout=0.5`, `weight_decay=1e-2`, `label_smoothing=0.1`
- Cosine + Plateau dual scheduler
- `max_patches=1024`, `patience=8`, `epochs=40`

**Result to compare:**  Quantum E2 → test AUC **0.6851**, val AUC 0.6507


In [1]:
# Cell 1 — Setup
import os, sys, time, json, gc
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'

sys.path.insert(0, '..')
from pathq.model_v2   import QuantaPathV2
from pathq.dataset_v2 import get_loaders_from_features

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT     = Path('..').resolve()
FEAT_DIR = Path('data') / 'features_uni'
CKPT_DIR = ROOT / 'checkpoints'
OUT_DIR  = ROOT / 'outputs'
CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('CPU only')
print(f'FEAT : {FEAT_DIR.resolve()}')
print(f'CKPT : {CKPT_DIR}')

/home/kabi/.conda/envs/pathq/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[model_v2] Using Transformer for global branch
GPU  : NVIDIA GeForce RTX 5060 Laptop GPU
VRAM : 8.1 GB
FEAT : /home/kabi/PATHQ--Quantum-Digital-Pathology-for-Whole-Slide-Image-Analysis/notebooks/data/features_uni
CKPT : /home/kabi/PATHQ--Quantum-Digital-Pathology-for-Whole-Slide-Image-Analysis/checkpoints


In [2]:
# Cell 2 — Load data  (221 labeled slides — dataset bug fixed)
train_loader, val_loader, test_loader = get_loaders_from_features(
    features_dir = FEAT_DIR,
    batch_size   = 4,
    k            = 8,
    seed         = 42,
    max_patches  = 3000,
)
print(f'max_patches : 3000')
print(f'Train : {len(train_loader)} batches')
print(f'Val   : {len(val_loader)} batches')
print(f'Test  : {len(test_loader)} batches')

  Skipped 112 unlabeled file(s) (e.g. test_*) — keeping 221 labeled slides
Split: train=154 (pos=77) val=33 (pos=17) test=34 (pos=17)
max_patches : 3000
Train : 39 batches
Val   : 9 batches
Test  : 9 batches


In [3]:
# Cell 3 — Training functions (same as E2)
SEP  = '═' * 65
DASH = '─' * 65

def train_one(model, loader, optimizer, device):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        torch.cuda.empty_cache()
        logits, _ = model(batch)
        loss      = F.cross_entropy(logits, batch.y.view(-1), label_smoothing=0.1)
        loss_val  = loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        del logits, loss
        torch.cuda.empty_cache()
        total += loss_val; n += 1
    return total / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    probs, labels, tl, n = [], [], 0.0, 0
    for batch in loader:
        batch     = batch.to(device)
        logits, _ = model(batch)
        tl       += F.cross_entropy(logits, batch.y.view(-1)).item()
        probs.extend(torch.softmax(logits, 1)[:, 1].cpu().tolist())
        labels.extend(batch.y.view(-1).cpu().tolist())
        n += 1
    p, l  = np.array(probs), np.array(labels)
    preds = (p >= 0.5).astype(int)
    auc   = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    f1    = f1_score(l, preds, zero_division=0)
    tp = int(((preds == 1) & (l == 1)).sum())
    fn = int(((preds == 0) & (l == 1)).sum())
    tn = int(((preds == 0) & (l == 0)).sum())
    fp = int(((preds == 1) & (l == 0)).sum())
    return {
        'auc'        : round(auc, 6),
        'f1'         : round(f1, 6),
        'loss'       : round(tl / max(n, 1), 6),
        'sensitivity': round(tp / max(tp + fn, 1), 4),
        'specificity': round(tn / max(tn + fp, 1), 4),
    }


def run_classical(model, tr, va, te, device,
                  ckpt_best, ckpt_latest,
                  epochs=40, lr=3e-5, patience=8):

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-2,
    )
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-7
    )
    plateau_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-7
    )

    best_auc, pat, start = 0.0, 0, 1

    if Path(ckpt_latest).exists():
        try:
            ck = torch.load(ckpt_latest, weights_only=False)
            model.load_state_dict(ck['model_state'])
            start    = ck['epoch'] + 1
            best_auc = ck['best_auc']
            for _ in range(start - 1):
                cosine_sched.step()
            print(f'Resumed from epoch {start-1}  best_auc={best_auc:.4f}')
        except Exception as e:
            print(f'Checkpoint incompatible — starting fresh. ({e})')
            start, best_auc = 1, 0.0
    else:
        print('No checkpoint found — starting fresh.')

    print()
    print(SEP)
    print(' E3 Classical GAT-Transformer — max_patches=3000 (FIXED data)')
    print(' dropout=0.5 | weight_decay=1e-2 | label_smoothing=0.1')
    print(f' patience={patience} | epochs={epochs}')
    print(SEP)
    print(f' {"Ep":>3} {"TrL":>8} {"VaL":>8} {"VaAUC":>7} {"VaF1":>7} {"LR":>9} {"Secs":>6}')
    print(DASH)

    for ep in range(start, epochs + 1):
        t0   = time.time()
        tl   = train_one(model, tr, optimizer, device)
        vm   = evaluate(model, va, device)

        cosine_sched.step()
        plateau_sched.step(vm['auc'])

        current_lr = optimizer.param_groups[0]['lr']
        flag       = ''

        if vm['auc'] > best_auc:
            best_auc = vm['auc']
            pat      = 0
            flag     = '*'
            torch.save({'model_state': model.state_dict(),
                        'epoch': ep, 'best_auc': best_auc}, ckpt_best)
        else:
            pat += 1

        torch.save({'model_state': model.state_dict(),
                    'epoch': ep, 'best_auc': best_auc}, ckpt_latest)

        ow   = ' ⚠overfit' if vm['loss'] > tl * 2.5 else ''
        secs = int(time.time() - t0)
        print(f' {ep:>3} {tl:>8.4f} {vm["loss"]:>8.4f} '
              f'{vm["auc"]:>7.4f} {vm["f1"]:>7.4f} '
              f'{current_lr:>9.2e} {secs:>5}s {flag}{ow}')

        if pat >= patience:
            print()
            print(f' Early stop ep {ep} — patience={patience}')
            break

        torch.cuda.empty_cache()
        gc.collect()

    ck = torch.load(ckpt_best, weights_only=False)
    model.load_state_dict(ck['model_state'])
    tm = evaluate(model, te, device)

    print(DASH)
    print(f' Best val AUC : {best_auc:.4f}')
    print(f' Test AUC     : {tm["auc"]:.4f}')
    print(f' F1           : {tm["f1"]:.4f}')
    print(f' Sensitivity  : {tm["sensitivity"]:.4f}')
    print(f' Specificity  : {tm["specificity"]:.4f}')
    print(SEP)
    return {**tm, 'val_auc': best_auc,
            'gap': round(tm['auc'] - best_auc, 6)}

print('Functions loaded ✓')

Functions loaded ✓


In [4]:
# Cell 4 — Run E3 Classical
CKPT_BEST   = str(CKPT_DIR / 'E3_classical_3000_best.pth')
CKPT_LATEST = str(CKPT_DIR / 'E3_classical_3000_latest.pth')

model_E3 = QuantaPathV2(
    use_vqc = False,
    in_dim  = 1040,
).to(DEVICE)

n_p = sum(p.numel() for p in model_E3.parameters() if p.requires_grad)
print(f'Trainable params : {n_p:,}')
print(f'use_vqc          : False  (classical GAT-Transformer only)')
print(f'in_dim           : 1040   (UNI 1024 + pos.enc 16)')
print()

result_classical = run_classical(
    model_E3,
    train_loader, val_loader, test_loader,
    DEVICE,
    ckpt_best   = CKPT_BEST,
    ckpt_latest = CKPT_LATEST,
    epochs  = 40,
    lr      = 3e-5,
    patience= 8,
)

with open(OUT_DIR / 'E3_classical_3000_result.json', 'w') as f:
    json.dump({
        'experiment'   : 'E3_classical_3000_fixed',
        'use_vqc'      : False,
        'max_patches'  : 3000,
        'dropout'      : 0.5,
        'weight_decay' : 1e-2,
        'label_smooth' : 0.1,
        'patience'     : 8,
        'data_fix'     : '221 labeled slides (333 -> 221, removed 112 test_*)',
        **result_classical,
    }, f, indent=2)

print('\nE3_classical_result.json saved.')

QuantaPathV2: use_vqc=False, trainable=1,227,906
Trainable params : 1,227,906
use_vqc          : False  (classical GAT-Transformer only)
in_dim           : 1040   (UNI 1024 + pos.enc 16)

No checkpoint found — starting fresh.

═════════════════════════════════════════════════════════════════
 E3 Classical GAT-Transformer — max_patches=3000 (FIXED data)
 dropout=0.5 | weight_decay=1e-2 | label_smoothing=0.1
 patience=8 | epochs=40
═════════════════════════════════════════════════════════════════
  Ep      TrL      VaL   VaAUC    VaF1        LR   Secs
─────────────────────────────────────────────────────────────────
   1   0.6992   0.6793  0.5110  0.5185  3.00e-05    90s *
   2   0.6641   0.6651  0.5331  0.5185  2.98e-05    88s *
   3   0.6429   0.6773  0.5588  0.5185  2.96e-05    88s *
   4   0.6300   0.7114  0.5515  0.5185  2.93e-05    88s 
   5   0.6253   0.7190  0.5772  0.5185  2.89e-05    89s *
   6   0.6078   0.7260  0.5956  0.5185  2.84e-05    89s *
   7   0.6086   0.7525  0.6140 

In [6]:
# Cell 5 — Compare: Classical (E3) vs Quantum (E2)
import json
from pathlib import Path

OUT = Path('..') / 'outputs'

# Load quantum E2 result
with open(OUT / 'E2_result.json') as f:
    res_q = json.load(f)

# Classical result from this run
res_c = result_classical

SEP = '=' * 68
print()
print(SEP)
print(' WEEK 3 RESULTS — CLEAN DATA (221 slides, max_patches=1024)')
print(SEP)
print(f'{"":<28} {"Classical":>12} {"Quantum":>12}')
print(f'{"":<28} {"GAT-Trans":>12} {"VQC+GAT":>12}')
print('-' * 55)
print(f'{"Val AUC":<28} {res_c["val_auc"]:>12.4f} {res_q["val_auc"]:>12.4f}')
print(f'{"Test AUC":<28} {res_c["auc"]:>12.4f} {res_q["auc"]:>12.4f}')
print(f'{"Val→Test Gap":<28} {res_c["gap"]:>+12.4f} {res_q["gap"]:>+12.4f}')
print(f'{"F1":<28} {res_c["f1"]:>12.4f} {res_q["f1"]:>12.4f}')
print(f'{"Sensitivity":<28} {res_c["sensitivity"]:>12.4f} {res_q["sensitivity"]:>12.4f}')
print(f'{"Specificity":<28} {res_c["specificity"]:>12.4f} {res_q["specificity"]:>12.4f}')
print(SEP)

# Save combined comparison
combined = {
    'data_note'    : 'FIXED: 221 labeled slides (removed 112 test_* unlabeled)',
    'max_patches'  : 1024,
    'classical'    : {
        'val_auc': res_c['val_auc'], 'test_auc': res_c['auc'],
        'gap': res_c['gap'], 'f1': res_c['f1'],
        'sensitivity': res_c['sensitivity'], 'specificity': res_c['specificity'],
    },
    'quantum_VQC'  : {
        'val_auc': res_q['val_auc'], 'test_auc': res_q['auc'],
        'gap': res_q['gap'], 'f1': res_q['f1'],
        'sensitivity': res_q['sensitivity'], 'specificity': res_q['specificity'],
    },
}
with open(OUT / 'week3_fixed_comparison.json', 'w') as f:
    json.dump(combined, f, indent=2)
print('Saved: outputs/week3_fixed_comparison.json')


 WEEK 3 RESULTS — CLEAN DATA (221 slides, max_patches=1024)
                                Classical      Quantum
                                GAT-Trans      VQC+GAT
-------------------------------------------------------
Val AUC                            0.6581       0.6507
Test AUC                           0.6920       0.6851
Val→Test Gap                      +0.0340      +0.0344
F1                                 0.7097       0.6875
Sensitivity                        0.6471       0.6471
Specificity                        0.8235       0.7647
Saved: outputs/week3_fixed_comparison.json
